# Build Your Own NIFTY 500 Point-in-Time (PIT) Dataset

This notebook reproduces the full pipeline described in the repo README, end to end, from public data sources only. Run the cells in order.

**What you'll end up with:** a single parquet file — one row per (company, trading day), with a flag telling you whether that company was *actually* a NIFTY 500 constituent on that exact date.

**Runtime:** the OHLCV download stage (Section 5) is the slow part — expect 30-60+ minutes for ~1,000 companies. Everything before that is fast (a few minutes).

**Requirements:** Python 3.9+, internet access. No API keys needed — everything here uses free, public endpoints.

## 0. Setup

In [ ]:
!pip install -q requests pandas pyarrow pypdf yfinance jugaad-data

In [ ]:
import os, re, csv, glob, time, html, warnings
from datetime import datetime, timedelta
import requests
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = "nifty500_pit_data"
os.makedirs(f"{DATA_DIR}/press_releases", exist_ok=True)
os.makedirs(f"{DATA_DIR}/press_releases_text", exist_ok=True)
os.makedirs(f"{DATA_DIR}/constituent_snapshots", exist_ok=True)
os.makedirs(f"{DATA_DIR}/ohlcv_yfinance", exist_ok=True)
os.makedirs(f"{DATA_DIR}/ohlcv_bhavcopy", exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
print("Ready.")

## 1. Scrape NIFTY 500 index-change press releases

NSE Indices publishes a press release for every constituent change. The full archive (back to 1998) is embedded client-side in one HTML page — a single request gets the whole list, no need to page through hundreds of months.

In [ ]:
MEDIA_URL = "https://www.niftyindices.com/media"

resp = requests.get(MEDIA_URL, headers=HEADERS, timeout=30)
html_text = resp.text
print("fetched", len(html_text), "bytes")

start = html_text.find('id="articleList"')
end = html_text.find("</ul>", start)
block = html_text[start:end]

HEADER_ITEM_RE = re.compile(
    r'<li data-publishdate="([^"]+)">\s*<span>([^<]+)</span>\s*<p>\s*<a href="([^"]+)"[^>]*>\s*([^<]+?)\s*</a>',
)
items = HEADER_ITEM_RE.findall(block)
print("total press releases found in archive:", len(items))

In [ ]:
def parse_release_date(s):
    try:
        return datetime.strptime(s.strip(), "%B %d, %Y")
    except ValueError:
        return None

def is_broad_500(title):
    return any(p in title for p in ["Nifty 500 Index", "CNX 500 Index", "S&amp;P CNX 500", "S&P CNX 500", "Nifty500 Index"])

def is_generic_multi_index(title):
    t = title.lower()
    return t.startswith("replacements in indices") or t.startswith("change in indices")

CUTOFF = datetime(2010, 1, 1)  # change this if you want a different start date
relevant = []
for month, date_str, url, title in items:
    d = parse_release_date(date_str)
    if d is None or d < CUTOFF:
        continue
    if is_broad_500(title) or is_generic_multi_index(title):
        relevant.append({"date": d.strftime("%Y-%m-%d"), "pdf_url": url.strip(), "title": html.unescape(title.strip())})

relevant.sort(key=lambda r: r["date"])
print("relevant press releases to download:", len(relevant))

In [ ]:
# download every relevant press release PDF
for r in relevant:
    fname = f"{DATA_DIR}/press_releases/{r['date']}_{r['pdf_url'].split('/')[-1]}"
    if os.path.exists(fname):
        continue
    full_url = "https://www.niftyindices.com" + r["pdf_url"]
    try:
        resp = requests.get(full_url, headers=HEADERS, timeout=30)
        if resp.status_code == 200 and len(resp.content) > 500:
            with open(fname, "wb") as f:
                f.write(resp.content)
    except Exception as e:
        print("failed:", r["pdf_url"], e)
    time.sleep(0.3)

pdf_files = glob.glob(f"{DATA_DIR}/press_releases/*.pdf")
print("PDFs downloaded:", len(pdf_files))

In [ ]:
# extract text from every PDF
from pypdf import PdfReader

for fp in pdf_files:
    base = os.path.splitext(os.path.basename(fp))[0]
    txt_path = f"{DATA_DIR}/press_releases_text/{base}.txt"
    if os.path.exists(txt_path):
        continue
    try:
        reader = PdfReader(fp)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
    except Exception as e:
        print("failed to parse:", fp, e)

print("text files:", len(glob.glob(f"{DATA_DIR}/press_releases_text/*.txt")))

## 2. Parse press releases into add/exclude events

Each press release lists companies being excluded/included under one or more index headers (e.g. `"S&P CNX 500 Index:"`, `"4) Nifty 500 Index"`). We only keep the sections that refer to the *broad* NIFTY 500 index, not thematic sub-indices like "Nifty500 Value 50".

In [ ]:
HEADER_RE = re.compile(
    r'(?:^|\n)\s*(?:\(?([a-zA-Z0-9]+)\)?[\.\)]\s*)?((?:NIFTY|Nifty|CNX|S&P\s*CNX)[A-Za-z0-9 &\.\-]{0,40}?)\s*(?:Index)?\s*:?\s*(?=\n)',
    re.MULTILINE
)
ROW_RE = re.compile(r'^\s*(\d+)\s+(.+?)\s+([A-Z0-9&\-\.]{2,20})\s*$')
ROW_RE_NOSYM = re.compile(r'^\s*(\d+)\s+(.+?)\s*$')

def normalize_index_name(raw):
    s = re.sub(r'\s+', ' ', raw).strip().rstrip('.').strip()
    s = re.sub(r'\s+index$', '', s, flags=re.IGNORECASE).strip()
    return s

def is_broad_500_name(name):
    return bool(re.fullmatch(r'(NIFTY|Nifty|S&P\s*CNX|CNX)\s*500', re.sub(r'\s+', ' ', name).strip()))

def extract_rows(block, action, results):
    label_re = re.compile(r'being\s+' + action + r'[a-z]*\s*:?', re.IGNORECASE)
    m = label_re.search(block)
    if not m:
        return
    started = False
    for line in block[m.end():].split('\n'):
        line = line.strip()
        if not line:
            if started: break
            continue
        if re.match(r'^Sr\.?\s*No\.?', line, re.IGNORECASE):
            started = True; continue
        if re.search(r'being\s+(excluded|included)', line, re.IGNORECASE):
            break
        rm = ROW_RE.match(line)
        if rm:
            started = True
            _, company, symbol = rm.groups()
            results.append({"action": action, "company_name": company.strip(), "symbol": symbol.strip()})
            continue
        rm2 = ROW_RE_NOSYM.match(line)
        if rm2 and not re.search(r'being\s+(excluded|included)', line, re.IGNORECASE):
            started = True
            _, company = rm2.groups()
            results.append({"action": action, "company_name": company.strip(), "symbol": ""})
            continue
        if started: break

def parse_press_release(text, is_dedicated500):
    headers = list(HEADER_RE.finditer(text))
    events = []
    for i, h in enumerate(headers):
        if not is_broad_500_name(normalize_index_name(h.group(2))):
            continue
        start = h.end()
        end = headers[i+1].start() if i+1 < len(headers) else len(text)
        block = text[start:end]
        extract_rows(block, "exclude", events)
        extract_rows(block, "include", events)
    if is_dedicated500 and not events:
        extract_rows(text, "exclude", events)
        extract_rows(text, "include", events)
    return events

print("parser defined.")

In [ ]:
all_events = []
for r in relevant:
    base = f"{r['date']}_{r['pdf_url'].split('/')[-1]}"
    base = os.path.splitext(base)[0]
    txt_path = f"{DATA_DIR}/press_releases_text/{base}.txt"
    if not os.path.exists(txt_path):
        continue
    text = open(txt_path, encoding="utf-8").read()
    is_dedicated = is_broad_500(r["title"])
    events = parse_press_release(text, is_dedicated)
    for e in events:
        e["press_release_date"] = r["date"]
    all_events.extend(events)

events_df = pd.DataFrame(all_events)
events_df.to_csv(f"{DATA_DIR}/events_raw.csv", index=False)
print("total events parsed:", len(events_df))
events_df.head()

## 3. Anchor on real historical constituent snapshots (Wayback Machine)

Press releases alone only capture ad-hoc, corporate-action-driven changes — routine market-cap-ranking reconstitution doesn't seem to generate the same kind of itemized announcement. So instead of trusting a single starting list plus event diffs, we pull **multiple real historical snapshots** of NSE's actual published constituent CSV via the Wayback Machine, across both domains that have hosted it over the years.

In [ ]:
def wayback_cdx_search(url_pattern, from_year, to_year):
    cdx_url = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": url_pattern, "output": "json",
        "from": str(from_year), "to": str(to_year), "limit": 300,
        "filter": "statuscode:200",
    }
    r = requests.get(cdx_url, params=params, timeout=30)
    rows = r.json()
    if len(rows) <= 1:
        return []
    header, *data = rows
    return [dict(zip(header, row)) for row in data]

# search across the domains/paths known to have hosted the constituent list over the years
candidates = []
for pattern in [
    "nseindia.com/content/indices/ind_cnx500list.csv",
    "nseindia.com/content/indices/ind_nifty500list.csv",
    "niftyindices.com/IndexConstituent/ind_nifty500list.csv",
]:
    candidates.extend(wayback_cdx_search(pattern, 2010, 2030))

print("total candidate snapshots found:", len(candidates))
for c in candidates[:5]:
    print(c["timestamp"], c["original"])

In [ ]:
def download_wayback_snapshot(timestamp, original_url, out_path):
    wb_url = f"https://web.archive.org/web/{timestamp}id_/{original_url}"
    r = requests.get(wb_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    content = r.content
    # some captures are gzip-compressed even after requests' auto-decompress; guard against that
    if content[:2] == b'\x1f\x8b':
        import gzip
        content = gzip.decompress(content)
    with open(out_path, "wb") as f:
        f.write(content)

downloaded = 0
for c in candidates:
    date_str = datetime.strptime(c["timestamp"], "%Y%m%d%H%M%S").strftime("%Y-%m-%d")
    out_path = f"{DATA_DIR}/constituent_snapshots/nifty500_{date_str}.csv"
    if os.path.exists(out_path):
        continue
    try:
        download_wayback_snapshot(c["timestamp"], c["original"], out_path)
        # sanity check: must be a real constituent list (has ISIN column, ~500 rows)
        text = open(out_path, encoding="utf-8", errors="ignore").read()
        if "ISIN" not in text or text.count("\n") < 100:
            os.remove(out_path)
            continue
        downloaded += 1
    except Exception:
        continue
    time.sleep(0.5)

snapshot_files = sorted(glob.glob(f"{DATA_DIR}/constituent_snapshots/*.csv"))
print(f"usable snapshots downloaded: {len(snapshot_files)}")
for f in snapshot_files:
    print(" ", f)

> **Note:** the number of usable snapshots you get depends on what the Wayback Machine happened to crawl — it will vary run to run and may be less complete than the 17 snapshots used in the reference build. The more snapshots you have, the tighter your membership-date confidence will be. Feel free to widen the `pattern` list above if you find additional URL variants NSE has used over the years.

## 4. Build the membership interval table

For each pair of consecutive snapshots, diff the ISIN sets to find who entered/exited. Two things to handle carefully:

- **ISIN migrations**: a company can get a new ISIN (e.g. a face-value-split-driven reissue) while staying continuously listed. Naively diffing ISIN sets makes this look like a fake exit+entry. We detect same-symbol/same-company-name pairs across a boundary and treat them as one continuous membership instead.
- **Confidence labels**: every interval's start/end date is labeled by how it was determined — exact (from a press release), tightly-bounded (snapshots close together), or loosely-bounded (snapshots far apart) — so nothing is presented as more precise than it is.

In [ ]:
def load_constituent_csv(path):
    lines = open(path, encoding="utf-8-sig", errors="ignore").readlines()
    start = 0
    for i, line in enumerate(lines):
        if line.strip().lower().startswith("company name"):
            start = i
            break
    rows = list(csv.DictReader(lines[start:]))
    out = {}
    for r in rows:
        isin = r.get("ISIN Code", "").strip()
        if isin:
            out[isin] = {"isin": isin, "symbol": r.get("Symbol", "").strip(), "company_name": r.get("Company Name", "").strip()}
    return out

def norm_symbol(s):
    return re.sub(r'\s+', '', s.strip().upper())

def norm_name(s):
    s = s.strip().lower()
    s = re.sub(r'\.', '', s)
    s = re.sub(r'\blimited\b', 'ltd', s)
    s = re.sub(r'\bltd\b\.?', 'ltd', s)
    s = re.sub(r'&', 'and', s)
    return re.sub(r'\s+', ' ', s).strip()

snapshots = []
for f in snapshot_files:
    date_str = os.path.basename(f).replace("nifty500_", "").replace(".csv", "")
    snapshots.append((datetime.strptime(date_str, "%Y-%m-%d"), load_constituent_csv(f)))
snapshots.sort(key=lambda x: x[0])
print("snapshots loaded:", len(snapshots))

In [ ]:
# index scraped events by normalized symbol for date lookups
events_by_symbol = {}
for _, row in events_df.iterrows():
    sym = norm_symbol(row["symbol"])
    if not sym:
        continue
    d = datetime.strptime(row["press_release_date"], "%Y-%m-%d")
    events_by_symbol.setdefault(sym, []).append((d, row["action"]))

def find_event_date(symbol, action, window_start, window_end):
    matches = [d for d, a in events_by_symbol.get(norm_symbol(symbol), []) if a == action and window_start <= d <= window_end]
    return min(matches) if matches else None

open_intervals = {}
closed_intervals = []
first_date, first_snap = snapshots[0]
for isin, info in first_snap.items():
    open_intervals[isin] = {**info, "start_date": first_date, "start_confidence": "left_censored"}

for i in range(1, len(snapshots)):
    prev_date, prev_snap = snapshots[i-1]
    curr_date, curr_snap = snapshots[i]
    prev_isins, curr_isins = set(prev_snap), set(curr_snap)
    exited, entered = prev_isins - curr_isins, curr_isins - prev_isins
    gap_conf = "snapshot_adjacent" if (curr_date - prev_date).days <= 220 else "long_gap"

    exited_by_key = {}
    for isin in exited:
        info = open_intervals[isin]
        exited_by_key.setdefault((norm_symbol(info["symbol"]), norm_name(info["company_name"])), []).append(isin)

    migrations = []
    for isin in list(entered):
        info = curr_snap[isin]
        key = (norm_symbol(info["symbol"]), norm_name(info["company_name"]))
        cands = exited_by_key.get(key, [])
        if cands:
            old_isin = cands.pop(0)
            migrations.append((old_isin, isin))
            exited.discard(old_isin); entered.discard(isin)

    for old_isin, new_isin in migrations:
        old_info = open_intervals.pop(old_isin)
        closed_intervals.append({**old_info, "end_date": curr_date, "end_confidence": f"isin_migration_to_{new_isin}"})
        open_intervals[new_isin] = {**curr_snap[new_isin], "start_date": old_info["start_date"], "start_confidence": f"isin_migration_from_{old_isin}"}

    for isin in exited:
        info = open_intervals.pop(isin)
        ev = find_event_date(info["symbol"], "exclude", prev_date, curr_date)
        end_date = ev if ev else prev_date + (curr_date - prev_date) / 2
        confidence = "press_release_exact" if ev else f"interpolated_{gap_conf}"
        closed_intervals.append({**info, "end_date": end_date, "end_confidence": confidence})

    for isin in entered:
        info = curr_snap[isin]
        ev = find_event_date(info["symbol"], "include", prev_date, curr_date)
        start_date = ev if ev else prev_date + (curr_date - prev_date) / 2
        confidence = "press_release_exact" if ev else f"interpolated_{gap_conf}"
        open_intervals[isin] = {**info, "start_date": start_date, "start_confidence": confidence}

    for isin in (prev_isins & curr_isins):
        open_intervals[isin]["symbol"] = curr_snap[isin]["symbol"]
        open_intervals[isin]["company_name"] = curr_snap[isin]["company_name"]

for isin, info in open_intervals.items():
    closed_intervals.append({**info, "end_date": None, "end_confidence": "still_active"})

intervals_df = pd.DataFrame(closed_intervals)
intervals_df["start_date"] = pd.to_datetime(intervals_df["start_date"]).dt.strftime("%Y-%m-%d")
intervals_df["end_date"] = pd.to_datetime(intervals_df["end_date"]).dt.strftime("%Y-%m-%d")
intervals_df = intervals_df[["isin","symbol","company_name","start_date","start_confidence","end_date","end_confidence"]]
intervals_df.to_csv(f"{DATA_DIR}/membership_intervals.csv", index=False)

print("total intervals:", len(intervals_df))
print("unique companies ever in NIFTY 500:", intervals_df["isin"].nunique())
print("currently active:", (intervals_df["end_confidence"]=="still_active").sum())

## 5. Resolve each company to its current trading symbol

Yahoo Finance keys on the *current* ticker only — historical symbol changes don't matter for the download, only where the company trades today. We build a symbol resolution table by grouping ISIN-migration chains and checking each against NSE's current listed-equity master.

In [ ]:
# NSE's live site blocks plain HTTP clients (Akamai bot protection). A recent
# Wayback snapshot of the full equity master works as a practical substitute.
equity_master_candidates = wayback_cdx_search("nseindia.com/content/equities/EQUITY_L.csv", 2020, 2030)
equity_master_candidates += wayback_cdx_search("nseindia.com/content/equities/equity_l.csv", 2020, 2030)
equity_master_candidates.sort(key=lambda c: c["timestamp"], reverse=True)

isin_to_current_symbol = {}
for c in equity_master_candidates[:1]:
    out_path = f"{DATA_DIR}/nse_equity_master.csv"
    download_wayback_snapshot(c["timestamp"], c["original"], out_path)
    reader = csv.DictReader(open(out_path, encoding="utf-8-sig", errors="ignore"))
    reader.fieldnames = [h.strip() for h in reader.fieldnames]
    for row in reader:
        isin = row.get("ISIN NUMBER", "").strip()
        sym = row.get("SYMBOL", "").strip()
        if isin:
            isin_to_current_symbol[isin] = sym
    print("equity master snapshot used:", c["timestamp"], "-", len(isin_to_current_symbol), "listed ISINs")

In [ ]:
# chain ISIN migrations together (same company, ISIN changed over time)
migration_target = {}
for _, row in intervals_df.iterrows():
    m = re.match(r"isin_migration_to_(\S+)", str(row["end_confidence"]))
    if m:
        migration_target[row["isin"]] = m.group(1)

def chain_head(isin):
    seen = set()
    while isin in migration_target and isin not in seen:
        seen.add(isin); isin = migration_target[isin]
    return isin

by_isin = intervals_df.groupby("isin")
chains = {}
for isin in by_isin.groups:
    chains.setdefault(chain_head(isin), []).append(isin)

manifest_rows = []
for head_isin, chain_isins in chains.items():
    recs = pd.concat([by_isin.get_group(i) for i in chain_isins])
    starts = pd.to_datetime(recs["start_date"])
    real_ends = pd.to_datetime(recs[~recs["end_confidence"].str.startswith("isin_migration_to_")]["end_date"])
    still_active = real_ends.isna().any() if len(real_ends) else True
    last_symbol = by_isin.get_group(head_isin).iloc[-1]["symbol"]
    current_symbol = isin_to_current_symbol.get(head_isin, "")

    manifest_rows.append({
        "isin": head_isin,
        "prior_isins_in_chain": ";".join(i for i in chain_isins if i != head_isin),
        "current_symbol": current_symbol or last_symbol,
        "yfinance_ticker": f"{(current_symbol or last_symbol)}.NS",
        "resolved_from_live_master": bool(current_symbol),
        "download_start_date": starts.min().strftime("%Y-%m-%d"),
        "download_end_date": "" if still_active else real_ends.max().strftime("%Y-%m-%d"),
        "still_active": still_active,
    })

manifest_df = pd.DataFrame(manifest_rows)
# drop temporary "DUMMY" demerger placeholder scrips -- not real independently-tradeable securities
manifest_df = manifest_df[~manifest_df["current_symbol"].str.startswith("DUMMY")]
manifest_df.to_csv(f"{DATA_DIR}/download_manifest.csv", index=False)

print("companies to download:", len(manifest_df))
print("resolved to a live current symbol:", manifest_df["resolved_from_live_master"].sum())

## 6. Download OHLCV

Primary source: **yfinance**. For companies with no live ticker to query (delisted/merged/acquired), fall back to NSE's own historical archive via `jugaad-data`, which handles NSE's session/cookie requirements properly.

**This is the slow step.** Expect 30-60+ minutes for the full universe.

In [ ]:
import yfinance as yf

def download_yfinance(ticker, start_date, end_date):
    end = (pd.Timestamp(end_date) + pd.Timedelta(days=1)) if end_date else (pd.Timestamp.today() + pd.Timedelta(days=1))
    df = yf.Ticker(ticker).history(start=start_date, end=end.strftime("%Y-%m-%d"), auto_adjust=False, actions=True)
    if df is None or df.empty:
        return None
    df = df.reset_index()
    # NOTE: yfinance's raw (auto_adjust=False) Close is ALREADY split-adjusted at
    # the source -- auto_adjust only controls dividend adjustment. Do not re-adjust.
    df = df.rename(columns={"Date":"date","Open":"open","High":"high","Low":"low","Close":"close",
                             "Volume":"volume","Dividends":"dividends","Stock Splits":"stock_splits"})
    return df[["date","open","high","low","close","volume","dividends","stock_splits"]]

ok, fail = 0, 0
fail_list = []
for _, row in manifest_df.iterrows():
    out_path = f"{DATA_DIR}/ohlcv_yfinance/{row['isin']}.parquet"
    if os.path.exists(out_path):
        ok += 1; continue
    df = download_yfinance(row["yfinance_ticker"], row["download_start_date"], row["download_end_date"])
    if df is not None and not df.empty:
        df["isin"] = row["isin"]; df["symbol"] = row["current_symbol"]
        df.to_parquet(out_path, index=False)
        ok += 1
    else:
        fail += 1
        fail_list.append(row.to_dict())
    time.sleep(0.3)

print(f"yfinance: ok={ok} fail={fail}")

In [ ]:
from jugaad_data.nse import stock_df

def download_bhavcopy(symbol, start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else datetime.today().date()
    try:
        df = stock_df(symbol=symbol, from_date=start, to_date=end, series="EQ")
    except Exception:
        return None
    if df is None or df.empty:
        return None
    df = df.rename(columns={"DATE":"date","OPEN":"open","HIGH":"high","LOW":"low","CLOSE":"close","VOLUME":"volume"})
    df["date"] = pd.to_datetime(df["date"])
    # NSE's historical archive (via jugaad-data) systematically mislabels Friday
    # sessions as the following Sat/Sun in some periods -- correct back to Friday.
    dow = df["date"].dt.dayofweek
    df.loc[dow == 5, "date"] -= pd.Timedelta(days=1)
    df.loc[dow == 6, "date"] -= pd.Timedelta(days=2)
    df = df.sort_values("volume", ascending=False).drop_duplicates(subset=["date"], keep="first")
    df["dividends"] = 0.0; df["stock_splits"] = 0.0
    return df[["date","open","high","low","close","volume","dividends","stock_splits"]].sort_values("date")

bhav_ok, bhav_fail = 0, 0
for row in fail_list:
    out_path = f"{DATA_DIR}/ohlcv_bhavcopy/{row['isin']}.parquet"
    if os.path.exists(out_path):
        bhav_ok += 1; continue
    df = download_bhavcopy(row["current_symbol"], row["download_start_date"], row["download_end_date"])
    if df is not None and not df.empty:
        df["isin"] = row["isin"]; df["symbol"] = row["current_symbol"]
        df.to_parquet(out_path, index=False)
        bhav_ok += 1
    else:
        bhav_fail += 1
    time.sleep(0.5)

print(f"bhavcopy fallback: ok={bhav_ok} fail={bhav_fail}")
print("(remaining failures are genuinely unreachable on either source -- typically very old delistings)")

## 7. Consolidate into the final PIT parquet

Merge both sources, join against the membership-interval table so every row carries the point-in-time constituent flag, and apply the extra cleanup steps that catch spurious low-volume single-share prints in the fallback source.

In [ ]:
all_frames = []
for _, row in manifest_df.iterrows():
    isin = row["isin"]
    chain = [isin] + [x for x in str(row["prior_isins_in_chain"]).split(";") if x]

    yf_path = f"{DATA_DIR}/ohlcv_yfinance/{isin}.parquet"
    bhav_path = f"{DATA_DIR}/ohlcv_bhavcopy/{isin}.parquet"
    if os.path.exists(yf_path):
        df = pd.read_parquet(yf_path); df["data_source"] = "yfinance"
        df["adjustment"] = "split_adjusted_no_dividend"
    elif os.path.exists(bhav_path):
        df = pd.read_parquet(bhav_path); df["data_source"] = "nse_bhavcopy_raw"
        df["adjustment"] = "raw_unadjusted"
        # drop spurious near-zero-volume prints wildly off the stock's trading range
        df = df.sort_values("date").reset_index(drop=True)
        prev_close = df["close"].shift(1)
        pct = (df["close"] - prev_close).abs() / prev_close
        df = df[~((df["volume"] < 200) & (pct > 0.40))].reset_index(drop=True)
    else:
        continue

    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
    combined_intervals = []
    for chain_isin in chain:
        sub = intervals_df[intervals_df["isin"] == chain_isin]
        for _, r in sub.iterrows():
            s = pd.Timestamp(r["start_date"])
            e = pd.Timestamp(r["end_date"]) if r["end_date"] and r["end_date"] != "NaT" else pd.Timestamp.today()
            combined_intervals.append((s, e))

    df["is_nifty500_constituent"] = df["date"].apply(lambda d: any(s <= d <= e for s, e in combined_intervals))
    all_frames.append(df[["isin","symbol","date","open","high","low","close","volume","dividends",
                           "stock_splits","is_nifty500_constituent","data_source","adjustment"]])

final_df = pd.concat(all_frames, ignore_index=True)
final_df = final_df[final_df["close"].notna()]  # drop any still-forming/incomplete session rows
final_df = final_df.drop_duplicates(subset=["isin","date"], keep="first")
final_df = final_df.sort_values(["isin","date"]).reset_index(drop=True)

OUT_PATH = f"{DATA_DIR}/nifty500_ohlcv_pit.parquet"
final_df.to_parquet(OUT_PATH, index=False)

print("FINAL DATASET")
print("path:", OUT_PATH)
print("rows:", len(final_df), " companies:", final_df['isin'].nunique())
print("date range:", final_df["date"].min(), "->", final_df["date"].max())

## 8. Validate

A quick sanity pass before trusting the output for backtesting.

In [ ]:
df = final_df

print("duplicate (isin,date) rows:", df.duplicated(subset=["isin","date"]).sum())
print("null prices:", df[["open","high","low","close"]].isna().sum().sum())
print("zero/negative prices:", (df[["open","high","low","close"]] <= 0).sum().sum())

bad_high = df["high"] < df[["open","low","close"]].max(axis=1)
bad_low = df["low"] > df[["open","high","close"]].min(axis=1)
print("OHLC internal-consistency violations:", int((bad_high | bad_low).sum()))

# the key sanity check: does the daily constituent count actually land near 500?
for d in pd.date_range(df["date"].min(), df["date"].max(), periods=6):
    d = d.normalize()
    n = df[(df["date"] == d) & (df["is_nifty500_constituent"])].shape[0]
    print(f"{d.date()}:  {n} constituents")

---

**Done.** `nifty500_pit_data/nifty500_ohlcv_pit.parquet` is your own reproduction of the dataset — filter on `is_nifty500_constituent` for any date to get the real historical NIFTY 500 universe, joined with clean OHLCV.

See the repo README for the full methodology writeup, known limitations, and adjustment-method reasoning.